# Pipeline de Pré-processamento — Dataset MoNuSeg

**Objetivo:** Carregar imagens histológicas do dataset MoNuSeg, executar o pipeline de pré-processamento (Cellpose + RGBA) e entender cada etapa do processo.

**O que você vai aprender:**
- Como o `MonusegDataset` carrega imagens e máscaras de ground truth
- Por que o Cellpose é usado como segmentação inicial
- O que é o `PreprocessingPipeline` e como ele orquestra os steps
- Como o `RGBAStep` constrói o tensor de 4 canais que alimenta a MarkerNet
- Como visualizar cada etapa do processamento

---

## 1. Configuração do Ambiente

Adicionamos a raiz do projeto ao `sys.path` para conseguir importar os módulos do `src/`. Isso é necessário porque o Jupyter roda a partir da pasta `notebooks/`.

In [ ]:
# # Clone the repository containing the project code
# import sys
# import os
# repo_dir = 'cell-fuzzy-seg'
# if not os.path.exists(repo_dir):
#     !git clone --branch homolog https://github.com/Pedro-io/cell-fuzzy-seg.git
# %cd {repo_dir}
# sys.path.append(f"/content/{repo_dir}")

Cloning into 'cell-fuzzy-seg'...
remote: Enumerating objects: 547, done.
remote: Counting objects: 100% (346/346), done.
remote: Compressing objects: 100% (226/226), done.
remote: Total 547 (delta 150), reused 283 (delta 90), pack-reused 201 (from 2)
Receiving objects: 100% (547/547), 232.07 MiB | 32.62 MiB/s, done.
Resolving deltas: 100% (221/221), done.
Updating files: 100% (214/214), done.
/content/cell-fuzzy-seg


In [3]:
!pip install -r requirements.txt --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 101.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 81.2 MB/s eta 0:00:00:00:0100:01


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# Projeto
from src.data.load.monuseg_dataset import MonusegDataset
from src.pipeline.preprocessing_pipeline import PreprocessingPipeline
from src.pipeline.steps.preprocessing.cellpose_step import CellposeStep
from src.pipeline.steps.preprocessing.rgba_step import RGBAStep

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Projeto: {project_root}")

ImportError: cannot import name 'PreprocessingPipeline' from 'src.pipeline.preprocessing_pipeline' (/content/cell-fuzzy-seg/src/pipeline/preprocessing_pipeline.py)

---
## 2. Conhecendo o Dataset MoNuSeg

O **MoNuSeg** (Multi-Organ Nuclei Segmentation) é um dataset de imagens histológicas com **núcleos celulares anotados manualmente**. Cada imagem possui uma máscara binária (ground truth) indicando quais pixels pertencem a núcleos.

### Estrutura dos dados

```
data_source/MoNuSegTrainingData/
├── Tissue_Images/          # Imagens .tif (1000×1000 pixels aproximadamente)
│   ├── TCGA-18-5592-01Z-00-DX1.tif
│   └── ...
├── Annotations/            # Máscaras em formato .xml (polígonos) ou .npy
│   ├── TCGA-18-5592-01Z-00-DX1.xml
│   └── ...
└── Binary_masks_instance/  # Máscaras binárias pré-processadas em .npy
```

> **Por que .xml?** As anotações originais do MoNuSeg estão em formato XML, onde cada `Region` contém vértices de um polígono que delimita um núcleo. O `MonusegDataset` faz o parsing desses polígonos e os rasteriza (via `cv2.fillPoly`) em uma máscara binária.

O `MonusegDataset` é carregado a partir de um arquivo YAML de configuração centralizado (`configs/datasets.yml`), que define:
- Onde estão as imagens e as máscaras
- As extensões dos arquivos
- Configurações de batch e pré-processamento

### 2.1 Carregar o Dataset

Vamos carregar o split de **treino** do MoNuSeg. O YAML carrega automaticamente as configurações.

In [ ]:
dataset = MonusegDataset(
    dataset_name="monuseg",
    config_key="monuseg_training",  # split de treino
    transform=None                   # sem augmentations por enquanto
)

print(f"Dataset: {dataset.get_dataset_name()}")
print(f"Split:   {dataset.get_config_key()}")
print(f"Tamanho: {len(dataset)} amostras")
print(f"\nImagens: {dataset.get_image_dir()}")
print(f"Máscaras: {dataset.get_mask_dir()}")
print(f"\nConfig do DataLoader:")
for k, v in dataset.get_loader_config().items():
    print(f"  {k}: {v}")

### 2.2 Inspecionar uma Amostra

Cada amostra retorna um dicionário com `id`, `image`, `ground_truth` e `meta`. Vamos ver os shapes e tipos.

In [ ]:
sample = dataset[0]

print(f"ID:          {sample['id']}")
print(f"\nKeys do sample:")
for k, v in sample.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:15s} | shape={str(v.shape):20s} | dtype={v.dtype} | min={v.min():.3f} | max={v.max():.3f}")
    elif isinstance(v, dict):
        print(f"  {k:15s} | dict com chaves: {list(v.keys())}")
    else:
        print(f"  {k:15s} | {type(v).__name__}: {v}")

### 2.3 Visualizar Imagem e Ground Truth

Vamos ver a imagem original lado a lado com sua máscara de ground truth.

> **Observação:** As imagens do MoNuSeg podem ser bem grandes (1000×1000+). O `ground_truth` é binário (0 = fundo, 1 = núcleo).

In [ ]:
def exibir_amostra(amostra):
    """Exibe imagem e ground truth lado a lado."""
    imagem = amostra['image']
    gt = amostra['ground_truth']
    
    # Se for 2D (grayscale), converte pra RGB para exibir
    if imagem.ndim == 2:
        img_display = np.stack([imagem] * 3, axis=-1)
    else:
        img_display = imagem[..., :3]
    
    # Normaliza para [0, 1]
    if img_display.max() > 1.0:
        img_display = img_display.astype(np.float32) / 255.0
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(img_display)
    axes[0].set_title(f"Imagem Original\n{imagem.shape}")
    axes[0].axis('off')
    
    axes[1].imshow(gt, cmap='gray')
    axes[1].set_title(f"Ground Truth (binário)\n{gt.shape}")
    axes[1].axis('off')
    
    # Overlay: imagem com máscara semi-transparente
    overlay = img_display.copy()
    mask_overlay = np.zeros_like(img_display)
    mask_overlay[..., 1] = gt  # canal verde
    overlay = overlay * 0.6 + mask_overlay * 0.4
    overlay = np.clip(overlay, 0, 1)
    
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay (imagem + gt verde)")
    axes[2].axis('off')
    
    plt.suptitle(f"Amostra: {amostra['id']}", fontsize=14)
    plt.tight_layout()
    plt.show()


exibir_amostra(sample)

---
## 3. Por que um Pipeline de Pré-processamento?

Antes de treinar a MarkerNet, precisamos preparar os dados. O pré-processamento envolve duas etapas principais:

### Etapa 1: Segmentação Inicial com Cellpose

O **Cellpose** é um modelo pré-treinado de segmentação de núcleos. Usamos ele para gerar uma **segmentação inicial** (máscaras de instância) que servirá como um dos canais de entrada da MarkerNet.

```
Imagem original  ──►  Cellpose  ──►  Máscara de segmentação (H, W)
```

> **Por que não usar só o Cellpose?** O Cellpose é genérico e pode não capturar bem as especificidades dos núcleos do MoNuSeg. A MarkerNet vai aprender a **refinar** essa segmentação inicial, gerando marcadores mais precisos.

### Etapa 2: Conversão para RGBA

A MarkerNet espera **4 canais** de entrada (RGBA):
- **R, G, B**: os 3 canais da imagem original
- **Alpha (A)**: a máscara de segmentação do Cellpose (1 onde há núcleo, 0 onde é fundo)

```
Imagem RGB (H, W, 3)  +  Máscara Cellpose (H, W, 1)  =  RGBA (H, W, 4)
```

> **Por que 4 canais?** A rede precisa ver **onde** o Cellpose detectou núcleos para aprender a ajustar os marcadores. O 4º canal funciona como uma "dica" (prior) para a MarkerNet.

---
## 4. Executando o Pipeline de Pré-processamento

O `PreprocessingPipeline` orquestra os steps em sequência. Cada step:
1. Recebe um dicionário de dados
2. Processa a informação
3. Adiciona novas chaves ao dicionário
4. Passa o dicionário para o próximo step

```
┌──────────────┐    ┌──────────────┐
│ CellposeStep │───►│  RGBAStep    │
│              │    │              │
│ Adiciona:    │    │ Adiciona:    │
│ segmentation │    │ rgba         │
│ flows        │    └──────────────┘
│ styles       │
└──────────────┘
```

### 4.1 Montar o Pipeline

Criamos um `PreprocessingPipeline` com os dois steps. A ordem importa: primeiro o Cellpose (que precisa da imagem), depois o RGBA (que precisa da imagem + segmentação).

In [ ]:
# Monta o pipeline de pré-processamento
pipeline = PreprocessingPipeline([
    CellposeStep(
        batch_size=1,
        pretreined_model="cpsam_v2",  # modelo Cellpose especializado em núcleos
        diam_mean=30.0,               # diâmetro médio esperado dos núcleos
        cellprob_threshold=0.0,       # threshold de probabilidade celular
        flow_threshold=0.2,           # threshold de fluxo
        min_size=4,                   # tamanho mínimo de objeto (remove ruídos)
    ),
    RGBAStep(),
])

print(f"Pipeline criado com {len(pipeline.steps)} steps:")
for i, step in enumerate(pipeline.steps, 1):
    print(f"  {i}. {step.name}")

### 4.2 Executar o Pipeline em uma Amostra

Agora vamos pegar uma amostra do dataset e passar pelo pipeline completo.

In [ ]:
# Pega uma amostra
amostra = dataset[0]

# Prepara o dicionário de entrada (só o que o pipeline precisa)
dados = {
    'image': amostra['image'],
    'id': amostra['id'],
}

# Executa o pipeline
print("Executando pipeline...")
resultado = pipeline.run(dados, verbose=True)
print("\nPipeline concluído!")

# Mostra as chaves adicionadas
print("\nChaves no resultado:")
for k, v in resultado.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:15s} | shape={str(v.shape):20s} | dtype={v.dtype} | min={v.min():.3f} | max={v.max():.3f}")
    else:
        print(f"  {k:15s} | {type(v).__name__}: {v}")

---
## 5. Visualizando Cada Etapa do Pipeline

Vamos executar step por step para ver o que cada um produz.

### 5.1 Step 1: CellposeStep

O Cellpose gera:
- **`segmentation`**: máscara de instância `(H, W)` — inteiros onde 0 = fundo, 1..N = cada núcleo
- **`flows`**: campos de fluxo (usados pelo Cellpose internamente)
- **`styles`**: vetores de estilo (representação global da imagem)

> **O que são `flows`?** O Cellpose não segmenta diretamente. Ele prediz um campo de vetores (flow) que apontam para o centro de cada núcleo. Depois, um algoritmo de tracking agrupa esses vetores em instâncias.

In [ ]:
# Step 1: Cellpose
dados_step1 = pipeline.steps[0].forward({'image': amostra['image']})
segmentation = dados_step1['segmentation']
flows = dados_step1['flows'] if 'flows' in dados_step1 else None

n_instancias = len(np.unique(segmentation)) - 1  # exclui background (0)
print(f"Núcleos detectados pelo Cellpose: {n_instancias}")
print(f"Segmentação: shape={segmentation.shape}, dtype={segmentation.dtype}")
print(f"Labels únicos: {np.unique(segmentation)}")

In [ ]:
# Visualizar a segmentação do Cellpose
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Imagem original
img = amostra['image']
if img.ndim == 2:
    img_display = np.stack([img] * 3, axis=-1)
else:
    img_display = img[..., :3].copy()
if img_display.max() > 1.0:
    img_display = img_display.astype(np.float32) / 255.0

axes[0].imshow(img_display)
axes[0].set_title("1. Imagem Original")
axes[0].axis('off')

# Ground truth
axes[1].imshow(amostra['ground_truth'], cmap='gray')
axes[1].set_title("2. Ground Truth (referência)")
axes[1].axis('off')

# Cellpose segmentation com colormap espectral
cmap = axes[2].imshow(segmentation, cmap='nipy_spectral')
axes[2].set_title(f"3. Cellpose ({n_instancias} núcleos)")
axes[2].axis('off')

plt.suptitle("Comparação: Ground Truth vs Cellpose", fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 Step 2: RGBAStep

O `RGBAStep` pega a imagem original + a máscara de segmentação e monta o tensor RGBA:

```
RGB (H, W, 3)  ──┐
                  ├──► RGBA (H, W, 4)
Alpha (H, W, 1) ──┘   [R, G, B, alpha]
```

O canal **alpha** é 1.0 onde há núcleo (segmentation > 0) e 0.0 onde é fundo. Isso informa a MarkerNet **onde** o Cellpose detectou células.

In [ ]:
# Step 2: RGBA
dados_step2 = pipeline.steps[1].forward(dados_step1)
rgba = dados_step2['rgba']

print(f"RGBA: shape={rgba.shape}, dtype={rgba.dtype}")
print(f"  Canal R: min={rgba[:,:,0].min():.3f}, max={rgba[:,:,0].max():.3f}")
print(f"  Canal G: min={rgba[:,:,1].min():.3f}, max={rgba[:,:,1].max():.3f}")
print(f"  Canal B: min={rgba[:,:,2].min():.3f}, max={rgba[:,:,2].max():.3f}")
print(f"  Alpha:   min={rgba[:,:,3].min():.3f}, max={rgba[:,:,3].max():.3f}")
print(f"  Pixels com alpha > 0: {(rgba[:,:,3] > 0).sum()} / {rgba[:,:,3].size}")

In [ ]:
# Visualizar os 4 canais do RGBA separadamente
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

canais = ['Red', 'Green', 'Blue', 'Alpha']
cmaps = ['Reds', 'Greens', 'Blues', 'gray']

for i, (nome, cmap) in enumerate(zip(canais, cmaps)):
    ax = axes[i]
    if i < 3:
        ax.imshow(rgba[:, :, i], cmap=cmap, vmin=0, vmax=1)
    else:
        # Alpha: mostra a máscara do Cellpose
        ax.imshow(rgba[:, :, 3], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"{nome}")
    ax.axis('off')

plt.suptitle("Os 4 Canais do RGBA (entrada da MarkerNet)", fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Visualização Comparativa Final

Vamos ver todo o fluxo lado a lado: imagem original → ground truth → Cellpose → RGBA.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Imagem original
axes[0].imshow(img_display)
axes[0].set_title("1. Imagem Original\n(input)", fontsize=11)
axes[0].axis('off')

# 2. Ground Truth
axes[1].imshow(amostra['ground_truth'], cmap='gray')
axes[1].set_title("2. Ground Truth\n(referência)", fontsize=11)
axes[1].axis('off')

# 3. Cellpose
axes[2].imshow(segmentation, cmap='nipy_spectral')
axes[2].set_title(f"3. Cellpose\n({n_instancias} instâncias)", fontsize=11)
axes[2].axis('off')

# 4. RGBA (visualização RGB + alpha como verde)
overlay_rgba = img_display.copy()
alpha_ch = rgba[:, :, 3]
overlay_rgba[..., 1] = np.clip(
    overlay_rgba[..., 1] + alpha_ch * 0.5, 0, 1
)
axes[3].imshow(overlay_rgba)
axes[3].set_title("4. RGBA (RGB + alpha verde)\n(input MarkerNet)", fontsize=11)
axes[3].axis('off')

plt.suptitle(f"Pipeline Completo — Amostra: {amostra['id']}", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Processando Várias Amostras

Agora que vimos o pipeline funcionando em uma amostra, vamos processar várias e ver as estatísticas.

In [ ]:
# Processa N amostras e coleta estatísticas
N_AMOSTRAS = 5

estatisticas = []

for idx in range(min(N_AMOSTRAS, len(dataset))):
    amostra = dataset[idx]
    dados = {'image': amostra['image'], 'id': amostra['id']}
    
    resultado = pipeline.run(dados, verbose=False)
    
    n_nucleos_gt = int(amostra['ground_truth'].sum())  # pixels de núcleo no gt
    n_instancias = len(np.unique(resultado['segmentation'])) - 1
    area_segmentada = (resultado['segmentation'] > 0).sum()
    
    estatisticas.append({
        'id': amostra['id'],
        'shape': amostra['image'].shape,
        'n_instancias_cellpose': n_instancias,
        'pixels_gt': n_nucleos_gt,
        'pixels_segmentados': area_segmentada,
    })
    
    print(f"[{idx+1}/{N_AMOSTRAS}] {amostra['id']:40s} | "
          f"shape={str(amostra['image'].shape):20s} | "
          f"instâncias Cellpose={n_instancias:2d} | "
          f"pixels GT={n_nucleos_gt:>8d}")

print(f"\nProcessadas {len(estatisticas)} amostras com sucesso.")

### 6.1 Visualizar Grid de Resultados

Vamos mostrar 3 amostras lado a lado: imagem → Cellpose → RGBA.

In [ ]:
N_VISUALIZAR = min(3, N_AMOSTRAS)
fig, axes = plt.subplots(N_VISUALIZAR, 3, figsize=(15, 5 * N_VISUALIZAR))

if N_VISUALIZAR == 1:
    axes = axes[None, :]

for row in range(N_VISUALIZAR):
    amostra = dataset[row]
    dados = {'image': amostra['image'], 'id': amostra['id']}
    resultado = pipeline.run(dados, verbose=False)
    
    # Prepara imagem para display
    img = amostra['image']
    if img.ndim == 2:
        img_disp = np.stack([img] * 3, axis=-1)
    else:
        img_disp = img[..., :3].copy()
    if img_disp.max() > 1.0:
        img_disp = img_disp.astype(np.float32) / 255.0
    
    axes[row, 0].imshow(img_disp)
    axes[row, 0].set_title(f"{amostra['id']}\nOriginal")
    axes[row, 0].axis('off')
    
    seg = resultado['segmentation']
    n_inst = len(np.unique(seg)) - 1
    axes[row, 1].imshow(seg, cmap='nipy_spectral')
    axes[row, 1].set_title(f"Cellpose\n({n_inst} instâncias)")
    axes[row, 1].axis('off')
    
    rgba = resultado['rgba']
    axes[row, 2].imshow(rgba)
    axes[row, 2].set_title("RGBA\n(4 canais)")
    axes[row, 2].axis('off')

plt.suptitle("Pipeline de Pré-processamento — Múltiplas Amostras", fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Usando o MonusegPreprocessedDataset

O `MonusegPreprocessedDataset` combina o dataset bruto com o pipeline de pré-processamento de forma transparente:

```python
dataset_bruto = MonusegDataset(...)
pipeline = PreprocessingPipeline([CellposeStep(), RGBAStep()])
dataset_processado = MonusegPreprocessedDataset(dataset_bruto, pipeline)

amostra = dataset_processado[0]
# amostra == {'image': ..., 'ground_truth': ..., 
#             'segmentation': ..., 'rgba': ..., 'processed': True}
```

Isso é útil porque:
1. O pré-processamento acontece **sob demanda** (lazy), só quando a amostra é acessada
2. O dataset processado pode ser passado para um `DataLoader` do PyTorch
3. Se você trocar o pipeline, o dataset se adapta automaticamente — sem modificar código

> **Importante:** O `MonusegPreprocessedDataset` nunca sabe quais steps estão no pipeline. Ele apenas chama `pipeline.run()`. Isso segue o **Princípio da Inversão de Dependência**: depende de uma abstração (o pipeline), não de implementações concretas.

### 7.1 Exemplo de Uso

Vamos criar o dataset processado e ver o que ele retorna.

In [ ]:
from src.data.load.monuseg_preprocessed_dataset import MonusegPreprocessedDataset

# Cria o dataset processado
dataset_processado = MonusegPreprocessedDataset(
    base_dataset=dataset,
    preprocessing_pipeline=pipeline,
)

print(f"Dataset processado: {len(dataset_processado)} amostras")

# Acessa uma amostra
amostra_processada = dataset_processado[0]

print("\nChaves disponíveis na amostra processada:")
for k, v in amostra_processada.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:15s} | shape={str(v.shape):20s} | dtype={v.dtype}")
    elif isinstance(v, dict):
        print(f"  {k:15s} | dict com chaves: {list(v.keys())}")
    else:
        print(f"  {k:15s} | {type(v).__name__}: {v}")

---
## 8. Resumo e Próximos Passos

### O que vimos neste notebook:

| Etapa | O que acontece | Saída produzida |
|-------|----------------|-----------------|
| **1. MonusegDataset** | Carrega imagem + anotação XML | `image` (H,W,C), `ground_truth` (H,W) |
| **2. CellposeStep** | Segmentação inicial com Cellpose | `segmentation` (H,W) — instâncias detectadas |
| **3. RGBAStep** | Concatena RGB + máscara Cellpose | `rgba` (H,W,4) — tensor de entrada da MarkerNet |
| **4. MonusegPreprocessedDataset** | Une dataset + pipeline | Amostras já pré-processadas |

### Por que essa arquitetura?

- **Separação de responsabilidades**: Cada step faz uma coisa só
- **Composição**: Steps podem ser adicionados/removidos sem alterar código existente
- **Reaproveitamento**: O Cellpose roda uma vez só, e o resultado fica disponível para todo o treinamento
- **Testabilidade**: Cada step pode ser testado isoladamente

### Próximos passos após o pré-processamento:

1. **Treinar a MarkerNet** — usar o RGBA como entrada para gerar marcadores fuzzy
2. **Implementar o MarkerStep** — encapsular a inferência da MarkerNet no pipeline
3. **Implementar a Rede Final de Segmentação** — usar os marcadores para segmentação refinada
4. **Integrar tudo no TrainingPipeline** — pipeline completo com backpropagation